# 5. Data Transformation, SMOTE & Save PKL

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
import joblib

In [ ]:
fraud_df = pd.read_csv('Fraud_Data_cleaned_featured.csv')
credit_df = pd.read_csv('creditcard_cleaned.csv')

In [ ]:
# Fraud data prep
cat_features = ['source', 'browser', 'sex', 'country']
num_features = ['purchase_value', 'age', 'hour_of_day', 'day_of_week',
                'time_since_signup_hours', 'user_transaction_count',
                'time_diff_prev_purchase_hours', 'user_avg_gap_hours',
                'device_usage_count', 'ip_usage_count']
X_fraud = fraud_df[cat_features + num_features]
y_fraud = fraud_df['class']
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

pipeline = Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42))])
X_train_f_res, y_train_f_res = pipeline.fit_resample(X_train_f, y_train_f)
X_test_f_trans = pipeline.named_steps['preprocessor'].transform(X_test_f)

In [ ]:
# Credit data prep
X_credit = credit_df.drop('Class', axis=1)
y_credit = credit_df['Class']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_credit, y_credit, test_size=0.2, stratify=y_credit, random_state=42)

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

smote = SMOTE(random_state=42)
X_train_c_res, y_train_c_res = smote.fit_resample(X_train_c_scaled, y_train_c)

In [ ]:
# Save as pickle
train_test_data = {
    'fraud': {'X_train': X_train_f_res, 'y_train': y_train_f_res,
              'X_test': X_test_f_trans, 'y_test': y_test_f},
    'credit': {'X_train': X_train_c_res, 'y_train': y_train_c_res,
               'X_test': X_test_c_scaled, 'y_test': y_test_c}
}
joblib.dump(train_test_data, 'train_test_sets.pkl')
print('Saved train_test_sets.pkl')